In [3]:
import pandas as pd
import numpy as np
from functools import reduce

In [4]:
# Reading files and dropping empty rows (for some reason there are some rows with only date)

# Denoted in USD
apple_ohlcv = pd.read_csv("AAPL.csv").dropna()
msci_emerging_ohlcv = pd.read_csv("MME=F.csv").dropna()
btc_ohlcv = pd.read_csv("BTC-USD.csv").dropna()
# Denoted in Euro
airbus_ohlcv = pd.read_csv("AIR.PA.csv").dropna()
lvmh_ohlcv = pd.read_csv("MC.PA.csv").dropna()
euro_stoxx_50_ohlcv = pd.read_csv("SC0D.DE.csv").dropna()
# IR and FX
euribor_3M = pd.read_csv("BundesBank Euribor 3M Daily.csv").dropna()
eur_usd = pd.read_csv("EURUSD=X.csv").dropna()

In [5]:
# Calculating Log Returns and Setting Proper dates

# US Denominated
apple_ohlcv['returns_aapl'] = np.log(apple_ohlcv['Adj Close'] / apple_ohlcv['Adj Close'].shift(1))
apple_ohlcv["Date"] = pd.to_datetime(apple_ohlcv["Date"], format='%Y-%m-%d')
msci_emerging_ohlcv['returns_msci'] = np.log(msci_emerging_ohlcv['Adj Close'] / msci_emerging_ohlcv['Adj Close'].shift(1))
msci_emerging_ohlcv["Date"] = pd.to_datetime(msci_emerging_ohlcv["Date"], format='%Y-%m-%d')
btc_ohlcv['returns_btc'] = np.log(btc_ohlcv['Adj Close'] / btc_ohlcv['Adj Close'].shift(1))
btc_ohlcv["Date"] = pd.to_datetime(btc_ohlcv["Date"], format='%Y-%m-%d')
# EUR Denominated
airbus_ohlcv['returns_airbus'] = np.log(airbus_ohlcv['Adj Close'] / airbus_ohlcv['Adj Close'].shift(1))
airbus_ohlcv["Date"] = pd.to_datetime(airbus_ohlcv["Date"], format='%Y-%m-%d')
lvmh_ohlcv['returns_lvmh'] = np.log(lvmh_ohlcv['Adj Close'] / lvmh_ohlcv['Adj Close'].shift(1))
lvmh_ohlcv["Date"] = pd.to_datetime(lvmh_ohlcv["Date"], format='%Y-%m-%d')
euro_stoxx_50_ohlcv['returns_eu50'] = np.log(euro_stoxx_50_ohlcv['Adj Close'] / euro_stoxx_50_ohlcv['Adj Close'].shift(1))
euro_stoxx_50_ohlcv["Date"] = pd.to_datetime(euro_stoxx_50_ohlcv["Date"], format='%Y-%m-%d')
# FX Rates
eur_usd['returns_EUR_USD'] = np.log(eur_usd['Adj Close'] / eur_usd['Adj Close'].shift(1))
eur_usd["Date"] = pd.to_datetime(eur_usd["Date"], format='%Y-%m-%d')
# Cleaning IR Data for Credit
euribor_3M["Date"] = pd.to_datetime(euribor_3M["Date"], format='%Y-%m-%d')
euribor_3M = euribor_3M[(euribor_3M["Date"] >= pd.to_datetime('2015-01-01')) & (euribor_3M["Date"] <= pd.to_datetime('2024-01-01'))]        # Keeping only the related dates


# Combining Returns into One Portfolio
portfolio = [euribor_3M[["Date", "EURIBOR_3M"]], eur_usd[["Date", "returns_EUR_USD"]], apple_ohlcv[['Date', 'returns_aapl']], msci_emerging_ohlcv[['Date', 'returns_msci']], btc_ohlcv[['Date', 'returns_btc']], airbus_ohlcv[['Date', 'returns_airbus']], lvmh_ohlcv[['Date', 'returns_lvmh']], euro_stoxx_50_ohlcv[['Date', 'returns_eu50']]]
portfolio = reduce(lambda left, right: pd.merge(left, right, on='Date', how='outer'), portfolio)    # Merge all DataFrames on the 'Date' column
portfolio = portfolio.sort_values(by='Date', ignore_index=True)     # Sorting based on date
portfolio = portfolio.fillna(0)                     # Setting non traded dates to have 0 return 
portfolio = portfolio.drop(portfolio.head(1).index) # Zeroth day has no returns
portfolio = portfolio.drop(portfolio.tail(1).index) # We do not need a first day of a new year
# We need to update EURUSD rates to be kept at the same as a previous close, not at zero
current_rate = 1.0
for i in range(portfolio.shape[0]):
    if portfolio.iloc[i, 1] != 0:
        current_rate = portfolio.iloc[i, 1]
    else:
        portfolio.iloc[i, 1] = current_rate
portfolio = portfolio.set_index('Date')                   # Setting Index to Dates
portfolio.index = pd.to_datetime(portfolio.index)
portfolio = portfolio.loc[~(portfolio[['returns_aapl', 'returns_msci', 'returns_airbus', 'returns_lvmh', 'returns_eu50']] == 0).all(axis=1)]        # Keep only rows where main assets were traded, i.e. delete weekend btc returns
portfolio.to_csv('_portfolio_returns.csv')       # Saving

In [6]:
portfolio

,EURIBOR_3M,returns_EUR_USD,returns_aapl,returns_msci,returns_btc,returns_airbus,returns_lvmh,returns_eu50
Date,,,,,,,,
2015-01-02,0.076,-0.000762,-0.009558,-0.014620,0.002489,0.004223,-0.010643,0.000000
2015-01-05,0.075,-0.011897,-0.028576,-0.017637,0.038169,0.003006,-0.029471,-0.037808
2015-01-06,0.073,-0.000620,0.000094,-0.005732,0.041796,0.022905,-0.011479,-0.007462
2015-01-07,0.070,-0.005346,0.013925,0.022415,0.028073,0.025714,0.000796,0.007727
2015-01-08,0.070,-0.003320,0.037703,0.017348,-0.038046,0.033509,0.030940,0.034898
...,...,...,...,...,...,...,...,...
2023-12-22,3.931,0.005510,-0.005563,-0.005324,0.002931,0.006620,-0.007697,-0.001098
2023-12-26,3.931,-0.000572,-0.002845,0.007779,-0.025374,0.000000,0.000000,0.000000
2023-12-27,3.925,0.002062,0.000518,0.005965,0.021462,-0.000143,-0.003123,0.002195
